<a href="https://colab.research.google.com/github/jolineuichanco/DataAnalytics/blob/main/misc/15_095_Recitation_7_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recitation 7: Deep Learning

In today's recitation, we will cover:
1. Feedforward neural networks (FNN)
2. Convolutional neural networks (CNN)
3. Improving training
4. Pretrained vision models - transfer learning and finetuning
5. Pretrained language models - transfer learning, feature embeddings, text generation, summarization
6. Multimodal models using feature embeddings

**Cats vs dogs dataset**

Throughout the first portion of today's recitation, we will be classifying images as dogs or cats.

**Library imports**

First, we import the necessary libraries for the cat/dog dataset's portion of the recitation.

In [ ]:
import os
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

tf.keras.utils.set_random_seed(42)

Now we load the image dataset.

In [ ]:
_URL = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
path_to_zip = tf.keras.utils.get_file("cats_and_dogs.zip", origin=_URL, extract=True)
PATH = os.path.join(os.path.dirname(path_to_zip), "cats_and_dogs_filtered")

train_dir = os.path.join(PATH, "train")
validation_dir = os.path.join(PATH, "validation")

#Choose your batch size: typically ranges from 16 to 256, using powers of 2.
BATCH_SIZE = 32

# Choose your image size: the larger, the finer the details on the image
# but also longer computational time and potential for overfit
# Classic sizes range from 160 to 512. 224 is a typical value.
# The choice depends also on the architecture you plan to use
IMG_SIZE = (160, 160)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

Since the original dataset doesn't contain a test set, we will create one. To do so, we determine how many batches of data are available in the validation set using `tf.data.experimental.cardinality` and move 20% of them to a test set.

In [ ]:
val_batches = tf.data.experimental.cardinality(validation_dataset)
test_dataset = validation_dataset.take(val_batches // 5)
validation_dataset = validation_dataset.skip(val_batches // 5)
print("Number of validation batches: %d" % tf.data.experimental.cardinality(validation_dataset))
print("Number of test batches: %d" % tf.data.experimental.cardinality(test_dataset))

In [ ]:
# Define some constants we will use throughout this recitation

MAX_RGB_VALUE = 255. # RGB values max out at 255, see: https://users.cs.utah.edu/~germain/PPS/Topics/color.html
INPUT_SHAPE = IMG_SIZE + (3,) # (160, 160, 3) = each image is 160x160, with 3 RGB channels

## 1) Feedforward Neural Network (FNN)

Recall that the activation functions are defined:

<img src="https://www.dropbox.com/scl/fi/6a8zd2cq5qoxx429m5vny/Screenshot-2024-10-14-at-6.13.03-PM.png?rlkey=jmqpr2fem67ni9chkxq9ki6fu&st=so2uirtq&dl=1" height=250 width="auto">

Also recall the fully-connected architecture from lecture:

<img src="https://www.dropbox.com/scl/fi/eyzsjtnxccmj0q1h1xuza/Screenshot-2024-10-14-at-6.10.36-PM.png?rlkey=8erwv0y26b8x5qrx1nzusqa71&st=atq4p31g&dl=1" height=250 width="auto">

We can define a feedforward network by its layers, and stack the layers ''sequentially'' using `tf.keras.Sequential`. Below we have have 7 layers, made of four types.
1. `tf.keras.layers.Input` defines the dimension of the input of the network
2. `tf.keras.layers.Rescaling` rescales all the input values by the argument, in this case by `1./MAX_RGB_VALUE` and therefore scaling all input values to [0,1] (improves numerical stability)
3. `tf.keras.layers.Flatten` flattens the input to a vector so that it can be accepted by the `tf.keras.layers.Dense` layers
4. `tf.keras.layers.Dense` are the fully-connected layers seen in class (see image below). We have defined all but the last dense layer to use the `ReLU` activation function, with different units (128, 64, 24, 10). The last layer uses the `sigmoid` ($\sigma$) activation function, which




In [ ]:
# Define the model architecture
model = tf.keras.Sequential([
    tf.keras.Input((160,160,3)),
    tf.keras.layers.Rescaling(1./MAX_RGB_VALUE),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(24, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Compile the model
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

# Train the model
model.fit(train_dataset, epochs=10, validation_data=validation_dataset)

# Evaluate the model on the test dataset
test_loss, test_acc = model.evaluate(test_dataset)
print("Test accuracy:", test_acc, "\tTest_loss:", test_loss)

## 2) Convolutional Neural Network (CNN)


Recall from lecture the CNN architecture:

<img src="https://www.dropbox.com/scl/fi/lwkmxz1c61yzwgdsmvzuk/Screenshot-2024-10-14-at-6.20.14-PM.png?rlkey=y43vnt0s20fwp5h7zl6kt84p8&st=7t9h5ceg&dl=1" height=250 width="auto">

There are several types of layers:
1. Convolutional layer - extracts features from an image by performing a convolution operation on it; the convolution operation converts all the pixels in its receptive field (what the kernel is covering) into a single value, decreasing image size and bringing all the information in the field together into a single pixel.
2. Max pooling layer - reduces the amount of information in an image while maintaining the essential features necessary for accurate image recognition
3. Flatten layer - reshapes the output of the previous convolutional block into a 1D vector, to be used for subsequent fully connected layers
4. Fully connected layer - performs the classification task


### More on the convolutional layer:
The most common type of convolution that is used is the 2D convolution layer and is usually abbreviated as conv2D. A filter or a kernel in a conv2D layer “slides” over the 2D input data, performing an elementwise multiplication. As a result, it will be summing up the results into a single output pixel. The kernel will perform the same operation for every location it slides over, transforming a 2D matrix of features into a different 2D matrix of features.

<img src="https://www.dropbox.com/scl/fi/ffclaodkw8jgpnwiddfbi/Screenshot-2024-10-15-at-7.51.58-AM.png?rlkey=c3vkp6p4752zehkasztppd76k&st=98w6t3ns&dl=1" height=250 width="auto">


Different paddings and strides can also be applied. Padding refers to the number of extra pixels of value `0` added to each border of the image. Stride refers to the number of pixels the filter moves across the image.

<img src="https://www.dropbox.com/scl/fi/xov3aa5z8n52fb7snnfbo/Screenshot-2024-10-15-at-7.54.03-AM.png?rlkey=qx7njmswspr2m6118wxor78oj&st=ysf6rljo&dl=1" height=250 width="auto">


This convolutional layer is instantiated in keras using the class [`tf.keras.layers.Conv2D`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D).


Remember that filters are learned through training and not preset. The filters are the `weights` in the convolutional layer. Some interesting filters are below:

<img src="https://www.dropbox.com/scl/fi/756ynw7hxppr55elum5s4/Screenshot-2024-10-15-at-8.01.11-AM.png?rlkey=b2mrwdt7isnw9srty1ovqnmwz&st=59nzoy5d&dl=1" height=250 width="auto">

[[credit](https://www.databricks.com/glossary/convolutional-layer)]






In [ ]:
# Define the model architecture
model = tf.keras.Sequential([
    tf.keras.Input((160,160,3)),
    tf.keras.layers.Rescaling(1./255.),
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Compile the model
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

# Train the model
model.fit(train_dataset, epochs=10, validation_data=validation_dataset)

# Evaluate the model on the test dataset
test_loss, test_acc = model.evaluate(test_dataset)
print("Test accuracy:", test_acc)

## 3) Improving training

Here we discuss some ways to improve training:
1. Data augmentation
2. Early stopping
3. Dropout

### 3.1 Data Augmentation

Data augmentation helps increase the diversity of your training set by applying random (but realistic) transformations. Some transformations include:
- [`RandomFlip`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/RandomFlip) - flips image horizontally and or vertically
- [`RandomRotation`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/RandomRotation)
- [`RandomContrast`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/RandomContrast) - randomly adjusts image contrast by  the provided factor
- [`RandomZoom`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/RandomZoom) - randomly zooms in/out independently on each axis of image

The complete list maybe be found [here](https://github.com/tensorflow/models/blob/master/research/object_detection/protos/preprocessor.proto)/[here](https://stackoverflow.com/questions/44906317/what-are-possible-values-for-data-augmentation-options-in-the-tensorflow-object).

[[credit](https://www.tensorflow.org/tutorials/images/data_augmentation)]


In [ ]:
# Data augmentation
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal"),
  tf.keras.layers.RandomRotation(0.2, fill_mode = "reflect"),
  tf.keras.layers.RandomContrast(0.5),
  tf.keras.layers.RandomZoom(.5, .2),
])

for image, _ in train_dataset.take(1):
  plt.figure(figsize=(10, 10))
  first_image = image[0]
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    augmented_image = data_augmentation(tf.expand_dims(first_image, 0))
    plt.imshow(augmented_image[0] / 255)
    plt.axis("off")


#### Two options for including data augmentation
Option 1: Make the preprocessing layers part of your model


In [ ]:
model = tf.keras.Sequential([
  # Add the preprocessing layers you created earlier.
  tf.keras.Input((160,160,3)),
  tf.keras.layers.Rescaling(1./255.),
  data_augmentation,
  tf.keras.layers.Conv2D(16, kernel=(3,3), activation="relu"),
  tf.keras.layers.MaxPooling2D(),
  # Rest of your model, etc.
])

**Option 2:** Apply the preprocessing layers to your dataset

In [ ]:
aug_train_dataset = train_dataset.map(
  lambda x, y: (data_augmentation(x, training=True), y))

### 3.2 Early Stopping

[Early stopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping) is a method to avoid overfitting. Early stopping will stop training when validation loss (or other metric) after a number of fluctuations in the metric. Intuitively, this stops training when the metric is at a (local) minimum.

In [ ]:
# Early stopping

# Define the model architecture
model = tf.keras.Sequential([
    tf.keras.Input((160,160,3)),
    tf.keras.layers.Rescaling(1./MAX_RGB_VALUE),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(24, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Compile the model
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

callback = tf.keras.callbacks.EarlyStopping(monitor="loss", patience=2) # typically val_loss

# Train the model
model.fit(train_dataset, epochs=20, validation_data=validation_dataset, callbacks=[callback])

# Evaluate the model on the test dataset
test_loss, test_acc = model.evaluate(test_dataset)
print("Test accuracy:", test_acc)

Only 6 epochs are run! Much less than 20.

### 3.3 Dropout

[`Dropout`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout) is another method to avoid overfiting. `Dropout` is a layer you can add to your model that randomly sets some neurons to zero. For example, it's common to add a rate of 50% for fully connected layers.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input((160,160,3)),
    tf.keras.layers.Rescaling(1./MAX_RGB_VALUE),
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(rate=0.5), ##### NEW DROPOUT LAYER #####
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Compile the model
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

# Train the model
model.fit(train_dataset, epochs=10, validation_data=validation_dataset)

# Evaluate the model on the test dataset
test_loss, test_acc = model.evaluate(test_dataset)
print("Test accuracy:", test_acc)

63.5% accuracy is a little better than the CNN without dropout (63.0%)!

## 4) Pretrained vision models

Instead of designing and training our own models from scratch, we can use pretrained models, which are trained on large corpora of data, and apply them to our dataset.

There are two ways we can apply these pretrained models:
1. Transfer learning - take the pretrained model as the base of our model, add a classification head (in our case), and only train the head layers. We are effectively transferring the ''features'' learned from the pretrained model as input to a simple FNN.
2. Finetuning - use our dataset to retrain some of the weights in the pretrained model (along with a classification head as above). This is called finetuning since we are ''customizing'' the model to our dataset.

### 4.1 Transfer learning

**Transfer learning** consists of taking features learned on one problem, and
leveraging them on a new, similar problem. For instance, features from a model that has
learned to identify animals in general may be useful to kick-start a model meant to identify birds only.

Transfer learning can be especially useful for tasks where your dataset has too little data to train a full-scale model from scratch.

The most common flavor of transfer learning in the context of deep learning is the following workflow:

1. Take layers from a previously trained model.
2. Freeze them, so as to avoid destroying any of the information they contain during future training rounds.
3. Add some new, trainable layers on top of the frozen layers. They will learn to turn the old features into predictions on a  new dataset.
4. Train the new layers on your dataset.
5. Optional: fine-tuning (to be discussed in the next section!)

**Typical transfer-learning workflow**

This leads us to how a typical transfer learning workflow can be implemented in Keras:

1. Instantiate a base model and load pre-trained weights into it.
2. Freeze all layers in the base model by setting `trainable = False`.
3. Create a new model on top of the output of one (or several) layers from the base model.
4. Train your new model on your new dataset.

#### 4.1.1 Step 1: Instantiate base model

Our base model is Google's MobileNet V2 model (designed for mobile devices so it's very fast!). It's pre-trained on the ImageNet dataset, a large research dataset consisting of 1.4M images and 1000 classes, including jackfruit, goldfish, or forklift. This base will help us classify cats and dogs for our specific dataset.

By specifying the include_top=False argument, you load a network that doesn't include the classification layers at the top, which is ideal for feature extraction since it retains more generality compared to the final/top layer.

**Additional info:**

Keras conveniently proposes to load many state-of-the-art and widely used models, including ResNet, EfficientNet, Inception,... Check here the full list: https://www.tensorflow.org/api_docs/python/tf/keras/applications

When you use a pretrained model, be careful to preprocess the data the way they expect it. In particular, check:

- if you need to standardize/normalize differently (in general we standardize using the ImageNet means and stds)
- if you need to reshape the picture (typically many models use the 224x224 size, but others can expect larger, like Xception with 299x299)
- in short, check the tf documentation of the model you want to use.

In [ ]:
# Step 1: Instantiate a base model and load pre-trained weights into it.
# Here, we create the base model from the pre-trained model MobileNet V2

base_model = tf.keras.applications.MobileNetV2(
    input_shape=INPUT_SHAPE, # Recall input shape is (160, 160, 3), since images are 160x160, with 3 RGB channels
    include_top=False, # Do not include the ImageNet classifier at the top.
    weights="imagenet" # Load weights pre-trained on ImageNet dataset
)

Let's take a closer look at the model. We can inspect the base model's output features and shape for an example batch of images...

We see that it converts each 160x160x3 image into a 5x5x1280 block of features.

In [ ]:
# Inspect the output of the model
image_batch, label_batch = next(iter(train_dataset))
feature_batch = base_model(image_batch)
print(feature_batch.shape)  # shape without pooling or flattening

#### 4.1.2 Step 2: Freeze all layers

We freeze our base model base in order to use it for transfer learning.

In [ ]:
# Step 2: Freeze all layers in the base model by setting `trainable=False`.
base_model.trainable = False # Freezes the base model.

Let's now inspect the architecture of the base model.

In [ ]:
# Inspect the base model architecture
base_model.summary()

#### 4.1.3 Create new model

Instead of using an explicit `tf.keras.layers.Rescaling` layer to preprocess the image, our base model has specific requirements for the input. In particular, it expects pixel values in [-1, 1]. We rescale them using the preprocessing method included with the model:

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
preprocess_input(image_batch).shape


We now add the classification head for our model. Since our base model gives us a block of features (spatial 5x5 spatial locations), we need to:
1. Use `tf.keras.layers.GlobalAveragePooling2D` to average over the features and convert them to a 1280-dimensional vector per image.
2. Apply a `tf.keras.layers.Dense layer` to convert the 1280-dimensional vector a single prediction per image. No activation function is needed here since the output itself is a prediction value; positive numbers predict class 1, negative numbers predict class 0. We specify this upon compiling the model.

In [ ]:
# 1. Convert feature block to vector
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
feature_batch_average = global_average_layer(feature_batch)
print(feature_batch_average.shape)

In [ ]:
# Convert vector to prediction
prediction_layer = tf.keras.layers.Dense(1)
prediction_batch = prediction_layer(feature_batch_average)
print(prediction_batch.shape)

In [ ]:
# Step 3: Create a new model on top of the output of one (or several) layers from the base model.

# Specify the input shape and preprocess them to be ready for the base model.
inputs = tf.keras.Input(shape=(160, 160, 3))
x = preprocess_input(inputs)

# Make sure the base_model is running in inference mode here,
# by passing `training=False`. This is important for fine-tuning, as you will
# learn in a few paragraphs.
# See note below on more about this!
x = base_model(x, training=False)

# Convert features of shape `base_model.output_shape[1:]` to vectors
x = global_average_layer(x)

# A Dense classifier with a single unit (binary classification)
# You can change the value to any number of your choice, depending on how many classes you have.
x = tf.keras.layers.Dropout(0.2)(x)

outputs = prediction_layer(x)

# Instantiate the model
model = tf.keras.Model(inputs, outputs)

In [ ]:
# Compile the model before training it. Since there are two classes,
# use the `tf.keras.losses.BinaryCrossentropy` loss with `from_logits=True`
# since the model provides a linear output.

base_learning_rate = 0.0001
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=["accuracy"])


In [ ]:
# The 2.5 million parameters in MobileNet are frozen, but there are 1.2 thousand trainable parameters
# in the Dense layer. These are divided between two `tf.Variable` objects, the weights and biases.

print("# trainable variables:", len(model.trainable_variables))

model.summary()

In [ ]:
initial_epochs = 10

loss0, accuracy0 = model.evaluate(test_dataset)
print("Initial performance: loss={:.2f}, accuracy={:.2f}".format(loss0, accuracy0))

#### 4.1.4 Train model

In [ ]:
# Step 4: Train your new model on your new dataset.
history = model.fit(train_dataset,
                    epochs=initial_epochs,
                    validation_data=(validation_dataset))

In [ ]:
loss10, accuracy10 = model.evaluate(test_dataset)
print("Trained model (transfer learning): loss={:.2f}, accuracy={:.2f}".format(loss10, accuracy10))

### 4.2 Finetuning

Recall that in pure transfer learning, you were only training additional layers on top of the base model. The weights of the base model were frozen, e.g. not updated during training.

**Fine-tuning** consists of unfreezing part of or the entire base model, and re-training it on the new data with a very low learning rate. The hope is to achieve meaningful improvements, by incrementally adapting (finetuning) the pretrained features to the new data.

**Useful notes**

Finetuning should only be done after training top-level classifier with the frozen pre-trained model. If you attempt to train all layers jointly, the updates will be too big and your base model will forget what it has learned on the pre-training dataset.

Finetuning should only be done on small number of top layers, rather than the entire model. In most convolutional networks, the higher up a layer is, the more specialized it is. Lower layers learn very simple and generalizable features. Higher layers have increasingly specific features for its training dataset. The goal of fine-tuning is to adapt the specialized features to work with the new dataset, rather than overwrite the generic learning.

#### 4.2.1 Unfreeze desired layers
Unfreeze the entire base model, then freeze the bottom layers to be un-trainable. This keeps the desired top layers unfrozen and trainable.

In [ ]:
base_model.trainable = True

In [ ]:
# Let's take a look to see how many layers are in the base model
print("Number of layers in the base model: ", len(base_model.layers))

# Fine-tune from this layer onwards
fine_tune_at = 100

# Freeze all the layers before the `fine_tune_at` layer
for layer in base_model.layers[:fine_tune_at]:
  layer.trainable =  False

#### 4.2.2 Recompile and train

Recompile the model, which is necessary for these changes to take effect and resume training.

In [ ]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer = tf.keras.optimizers.RMSprop(learning_rate=base_learning_rate/10),
              metrics=['accuracy'])

In [ ]:
print("# trainable variables:", len(model.trainable_variables))

model.summary()

In [ ]:
# Continue training the model
fine_tune_epochs = 10
total_epochs =  initial_epochs + fine_tune_epochs

history_fine = model.fit(train_dataset,
                         epochs=total_epochs,
                         initial_epoch=history.epoch[-1],
                         validation_data=validation_dataset)

In [ ]:
loss20, accuracy20 = model.evaluate(test_dataset)
print("Trained model (finetuning): loss={:.2f}, accuracy={:.2f}".format(loss20, accuracy20))

**Some important notes**

**`BatchNormalization` layers**
Many models contain `tf.keras.layers.BatchNormalization layers`. You can look for these in the model summary further up. This layer is a special case and precautions should be taken in the context of fine-tuning.

When you set `layer.trainable = False`, the `BatchNormalization` layer will run in inference mode, and will not update its mean and variance statistics.

When you unfreeze a model that contains `BatchNormalization `layers in order to do fine-tuning, you should keep the `BatchNormalization` layers in inference mode by passing `training = False` when calling the base model. Otherwise, the updates applied to the non-trainable weights will destroy what the model has learned.

For more details, see the [Transfer Learning Guide](https://www.tensorflow.org/guide/keras/transfer_learning).

**Validation vs training gap**
If you are wondering why the validation metrics are clearly better than the training metrics, the main factor is because layers like `tf.keras.layers.BatchNormalization` and `tf.keras.layers.Dropout` affect accuracy during training. They are turned off when calculating validation loss.

To a lesser extent, it is also because training metrics report the average for an epoch, while validation metrics are evaluated after the epoch, so validation metrics see a model that has trained slightly longer.

## 5) Language models
Now let's discuss language models. We can again create our own neural networks and train them on a specific language dataset, but we will go ahead and use pretrained large language models (LLM) for this section.

Pretrained models are very useful, as they are trained on ~the entire internet. Sometimes, domain-specific data is used in order to get specialized models; for example, [`ClinicalBERT`](https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT).

Some uses include:
1. Feature embeddings
2. Text generation
3. Summarization

You can find many pretrained models:
1. Tensorflow - tensorflow_hub, https://www.kaggle.com/models?tfhub-redirect=true
2. Pytorch - huggingface, https://huggingface.co/

Note that starting from this section onwards, we use PyTorch.

### 5.1 Pretrained models: feature embeddings

Feature embeddings are vector representations of a word, sentence, or block of text. For these LLMs, these vectors are spatial representations, which means that the closer two text-vectors are in $n$-dimensional space, the ''closer'' the meaning of the corresponding texts are in meaning.

These embeddings can be used as input into other models, but here, we show two methods of extracting embeddings from LLMs.

Note that an alternative, more lightweight workflow could also be:

1. Instantiate a base model and load pre-trained weights into it.
2. Run your new dataset through it and record the output of one (or several) layers
 from the base model. This is called **feature extraction**.
3. Use that output as input data for a new, smaller model.

A key advantage of that second workflow is that you only run the base model once on
 your data, rather than once per epoch of training. So it's a lot faster & cheaper. You can also use the features extracted for a completely different machine learning model or even task. Personally, I have had great success extracting features with neural nets and inputing them into an XGBoost model for example. It can be very practical for multimodal machine learning.

However, an issue with that second workflow is that it doesn't allow you to dynamically
modify the input data of your new model during training, which is required when doing
data augmentation, for instance. Transfer learning is typically used for tasks when
your new dataset has too little data to train a full-scale model from scratch, and in
such scenarios data augmentation is very important. So in what follows, we will focus
 on the first workflow.

We first install/import library packages.

In [ ]:
!pip install transformers
!pip install sacremoses

from transformers import BertModel, BertTokenizer, pipeline, set_seed
import torch

**Method 1**: The first method is manual. The steps are:
1. Load model and tokenizer
2. Tokenize the input text into tokens (more on tokens [here](https://learn.microsoft.com/en-us/dotnet/ai/conceptual/understanding-tokens))
3. Pass the input tokens into the model and obtain the embeddings by extracting the output of the penultimate layer in the model and reshaping the vector as needed.

In [ ]:
# 1. Load model
model = BertModel.from_pretrained("bert-base-uncased")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# 2. Tokenize
text = "A large language model's favorite ice cream is sherbert" # 🥁
tokens = tokenizer(text, return_tensors="pt")
print(tokens)
print("# Tokens:", len(tokens.input_ids[0]))

In [ ]:
# 3. Pass into model and extract embedding
with torch.no_grad():
    outputs = model(**tokens)
    embeddings = outputs.last_hidden_state[0]
    print("---Token embeddings (e.g. last hidden layer)---")
    print(embeddings.detach().numpy().shape)
    print(embeddings)
    print("---Entire text's embeddings (e.g. final layer)---")
    print(outputs.pooler_output.detach().numpy().shape)
    # print(outputs.pooler_output)

**Method 2**: The second method is more automatic. The steps are:
1. Instantiate a feature extactor, specifying the model you want
2. Pass the input text into the extractor
3. If you need embeddings for the entire text block, average the embeddings of each token and reshape accordingly


In [ ]:
medical_text = """
The SARS-CoV-2 virus can infect a wide range of cells and systems of the body.
COVID‑19 is most known for affecting the upper respiratory tract (sinuses, nose, and throat)
and the lower respiratory tract (windpipe and lungs). The lungs are the organs
most affected by COVID‑19 because the virus accesses host cells via the receptor for
the enzyme angiotensin-converting enzyme 2 (ACE2), which is most abundant on the surface
of type II alveolar cells of the lungs. The virus uses a special surface glycoprotein
called a "spike" to connect to the ACE2 receptor and enter the host cell.
"""

# 1. Instantiate feature extractor
feature_extractor = pipeline("feature-extraction", model="microsoft/biogpt")#, device="cuda")

In [ ]:
# 2. Pass input text
data = feature_extractor(text)
print(data)
print(np.array(data).shape)

In [ ]:
# 3. Embeddings for entire text block
# Reducing along the first dimension to get a 1024-dimensional array
pooler_output = np.array(data[0]).mean(axis=0)
print(pooler_output)
print(pooler_output.shape)

### 5.2 Pretrained models: text generation

We can also use pipline to generate text! We can give it a specific text prompt, and the LLM will "fill-in-the-blank."

In [ ]:
set_seed(42)

generator = pipeline("text-generation", model="gpt2")

In [ ]:
text_prompt = """The mission of the MIT Sloan School of Management is
to develop principled, innovative leaders who """
result = generator(text_prompt, max_length=30, num_return_sequences=1)

In [ ]:
print(result[0]["generated_text"])

### 5.3 Pretrained models: summarization



Note that previously we specified the model name in our `pipeline` object. Here, we haven't and instead we use some default model from Pytorch. We also show an example of [google-pegasus-xsum](https://huggingface.co/google/pegasus-xsum) below, where we can see how models differ!

You can find plenty of summarization models [here](https://huggingface.co/models?pipeline_tag=summarization).

In [ ]:
## using pipeline and task name to load a summarizer from a pre-trained model

summarizer = pipeline("summarization")

In [ ]:
article = """The only thing crazier than a guy in snowbound Massachusetts
boxing up the powdery white stuff
and offering it for sale online? People are actually buying it.
For $89, self-styled entrepreneur Kyle Waring will ship you 6 pounds
of Boston-area snow in an insulated Styrofoam box –
enough for 10 to 15 snowballs, he says.
But not if you live in New England or surrounding states.
His website and social media accounts claim to have filled more than 133 orders
for snow – more than 30 on Tuesday alone, his busiest day yet.
With more than 45 total inches, Boston has set a record this winter
for the snowiest month in its history. Most residents see the huge piles
of snow choking their yards and sidewalks as a nuisance, but Waring saw an
opportunity. According to Boston.com, it all started a few weeks ago,
when Waring and his wife were shoveling deep snow from their yard in
Manchester-by-the-Sea, a coastal suburb north of Boston. He joked about
shipping the stuff to friends and family in warmer states, and an
idea was born. His business slogan: “Our nightmare is your dream!”
At first, ShipSnowYo sold snow packed into empty 16.9-ounce water
bottles for $19.99, but the snow usually melted before it
reached its destination.
"""

In [ ]:
result = summarizer(article, max_length=30, min_length=30, num_return_sequences=4) #generates the summarization

In [ ]:
result[0]["summary_text"]

In [ ]:
summarizer = pipeline("summarization", model = "google/pegasus-xsum") # change the model to your favorite one

In [ ]:
result = summarizer(article, max_length=30, min_length=30, num_return_sequences=4) #generates the summarization
result[0]["summary_text"]

The result is quite different! You can do a deep-dive [here](https://huggingface.co/docs/transformers/task_summary#summarization) for low-level usage and task-specific fine-tuning.

## 6) Multimodal models

Recall multimodal ML trains models using data from multiple
modalities, including:
- Structured (Tabular) Data
- Image Data
- Language Data
- Time Series Data
- Etc.

One idea of incorporating multiple modalities is to extract features for each modality (using pre-
trained NNs or build your own) and then combine by concatenating the features into one vector.

To see how we can train a multimodal model, we will use tabular, language, and image data from the [PetFinder.my Adoption Prediction](https://www.kaggle.com/c/petfinder-adoption-prediction/) competition from Kaggle.

**A little bit about this dataset...**

Millions of stray animals suffer on the streets or are euthanized in shelters every day around the world. If homes can be found for them, many precious lives can be saved — and more happy families created.

PetFinder.my has been Malaysia’s leading animal welfare platform since 2008, with a database of more than 150,000 animals. Since animal adoption rates are strongly correlated to the metadata associated with their online profiles, such as descriptive text and photo characteristics, the goal of this dataset is to develop a model to predict the adoptability of pets. More specifically, how quickly is a pet adopted?

This is a multiclassification task, since the adoption speed is provided as a 4-class target:
- 0 - Pet was adopted on the same day as it was listed.
- 1 - Pet was adopted between 1 and 7 days (1st week) after being listed.
- 2 - Pet was adopted between 8 and 30 days (1st month) after being listed.
- 3 - Pet was adopted between 31 and 90 days (2nd & 3rd month) after being listed.
4-  - No adoption after 100 days of being listed. (There are no pets in this dataset that waited between 90 and 100 days).

In [ ]:
# Library import
import os
import pickle
from glob import glob
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn import model_selection
from transformers import pipeline
import torch
import torchvision
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from copy import deepcopy
import itertools

seed = 15095

In [ ]:
# Load the raw data and examine some rows
df_raw = pd.read_csv("https://www.dropbox.com/scl/fi/c4wibhaf2mpi8htmtof1w/df_v2.csv?rlkey=aljb22czvovkhwbb4fkszy07k&st=mh99e3s8&dl=1")
df_raw.head(5)

#### 6.1 Language feature extraction

We extract feature models using the `pipeline` method that we saw earlier. We provide the code for all datapoints, but also for a specific example to examine more closely.

In [ ]:
feature_extractor = pipeline("feature-extraction", model="gpt2") #, device="cuda")

In [ ]:
# Specific example
i = 0
text_col = "Description"
row = df_raw.iloc[i]
text = row[text_col]
data = feature_extractor(text)
# Reducing along the first dimension to get a 1024-dimensional array
pooler_output = np.array(data[0]).mean(axis=0)
print(pooler_output.shape)

In [ ]:
# Run for all datapoints
all_text_embeddings = []
text_col = "Description"
for i in tqdm(range(len(df_raw))):
    row = df_raw.iloc[i]
    text = row[text_col]
    data = feature_extractor(text)
    # Reducing along the first dimension to get a 1024-dimensional array
    pooler_output = np.array(data[0]).mean(axis=0)
    all_text_embeddings.append(pooler_output)

#### 6.2 Image feature extraction

We extract features from the pretrained model, `Resnet`. Note that any other model may be used here!

Again, we provide the code for all datapoints, but also for a specific example to examine more closely.

In [ ]:
# Load data into the Colab
!wget q --show-progress --no-check-certificate 'https://www.dropbox.com/scl/fi/vts2e79iv90i9ajhgdt9j/imgs_resized.pickle?rlkey=lu81p2oxdrz14sebjk8j3bzpl&st=74zn97mt&dl=1' -O pet-adoption-imgs.pickle


In [ ]:
# Load image data as RGB pixels
with open("/content/pet-adoption-imgs.pickle", "rb") as input_file:
    all_imgs_resized_arr = pickle.load(input_file)

In [ ]:
# Load a pre-trained ResNet model (or any other model)
resnet = torchvision.models.resnet50()

# Remove the final classification layer to get embeddings
# The last fully connected layer in ResNet is 'fc', which we'll remove
model = torch.nn.Sequential(*list(resnet.children())[:-1])

In [ ]:
# Specific example
img = all_imgs_resized_arr[0]
torch_img = torch.tensor(img).permute([2,0,1]).unsqueeze(0).float()

# Extract embeddings
with torch.no_grad():
    embeddings = model(torch_img)

# Flatten embeddings from [batch_size, num_channels, 1, 1] to [batch_size, num_channels]
embeddings = embeddings.view(embeddings.size(0), -1)

embeddings_np = embeddings.numpy()[0]
print(embeddings_np.shape)

In [ ]:
# Run for all examples
all_img_embeddings = []
for img in tqdm(all_imgs_resized_arr):
    torch_img = torch.tensor(img).permute([2,0,1]).unsqueeze(0).float()

    # Extract embeddings
    with torch.no_grad():
        embeddings = model(torch_img)

    # Flatten embeddings from [batch_size, num_channels, 1, 1] to [batch_size, num_channels]
    embeddings = embeddings.view(embeddings.size(0), -1)

    embeddings_np = embeddings.numpy()[0]
    all_img_embeddings.append(embeddings_np)

#### 6.2.3 Combine and train the HAIM model

Here, we will combine our modalities and train a HAIM XGBoost model.

In [ ]:
# Prepare modalities for combining
tab_df = df_raw.drop(["PetID", "Description"], axis=1)

all_img_embeddings_np = np.array(all_img_embeddings)
img_df = pd.DataFrame(all_img_embeddings_np, columns=[f"img_{i}" for i in range(all_img_embeddings_np.shape[1])])

all_text_embeddings_np = np.array(all_text_embeddings)
text_df = pd.DataFrame(all_text_embeddings_np, columns=[f"text_{i}" for i in range(all_text_embeddings_np.shape[1])])

In [ ]:
# Combine and check for duplication
df_combined = pd.merge(tab_df, img_df, left_index=True, right_index=True, how="left")
df_combined = pd.merge(df_combined, text_df, left_index=True, right_index=True, how="left")
len(df_combined), len(df_raw), len(img_df), len(text_df)

In [ ]:
# Extract features and target from dataset
target_col = "AdoptionSpeed"
X = df_combined.drop(target_col, axis=1)
y = df_combined[target_col]

In [ ]:
# Split dataset into train, validation, test
X_trainvalid, X_test, y_trainvalid, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=seed
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainvalid, y_trainvalid, test_size=0.15, stratify=y_trainvalid, random_state=seed
)

In [ ]:
# Encode categorical features as one-hot features

categorical_cols = [ # keep colors continuous since they are kind of on gradient
    "Type",
    "Gender",
    "MaturitySize",
    "FurLength",
    "Vaccinated",
    "Dewormed",
    "Sterilized",
    "Health",
    "MixedBreed",
]

def encode_categorical(df, categorical_cols):
    df_encoded = deepcopy(df)
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
    return df_encoded, label_encoders



# Encode categorical columns for train, valid, and test sets
X_train_encoded, le_train = encode_categorical(X_train, categorical_cols)
X_valid_encoded, le_valid = encode_categorical(X_valid, categorical_cols)
X_test_encoded, le_test = encode_categorical(X_test, categorical_cols)


In [ ]:
# Define parameter gride
lr_params = [0.05, 0.1, 0.2, 0.5]
max_depth_params = [3, 5, 7]
subsample_params = [0.8, 1.0]
colsample_params = [0.8, 1.0]

param_grid = {
    "max_depth": max_depth_params,
    "learning_rate": lr_params,
    "subsample": subsample_params,
    "colsample_bytree": colsample_params,
}


# Finetune hyperparameters via grid search
valid_score = []
clf_lis = []

for lr, max_depth in tqdm(list(itertools.product(lr_params, max_depth_params))):
        clf = XGBClassifier(learning_rate=lr,
                            max_depth=max_depth,
                            random_state=1) #, gpu_id=1, verbosity = 0)
        clf.fit(X_train, y_train)

        y_valid_pred = clf.predict(X_valid)
        valid_score.append(accuracy_score(y_valid, y_valid_pred))
        clf_lis.append(clf)

# Obtain the best model and output the correspondign accuracy score
best_clf_xgb = clf_lis[np.argmax(valid_score)]
best_acc_xgb = valid_score[np.argmax(valid_score)]
y_pred = best_clf_xgb.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"Acc on test set:", test_acc)


This might not seem very great, but on the Kaggle competition leaderboard, the best score is ~0.45 (albeit based on a different but similar metric). Now let's compare with a baseline model that just predicts the most common class:

In [ ]:
y_test.value_counts()

In [ ]:
223/len(y_test)

36.2% compared to 27.9% - so some improvement!